<a href="https://colab.research.google.com/github/samathasrikamireddy/Infosys_FreightQuote_AI/blob/main/Login__Page.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly


In [ ]:
import os
import random
import time
import sqlite3
import bcrypt
import jwt
from pyngrok import ngrok
import datetime
import smtplib
import subprocess
from email.mime.text import MIMEText
import streamlit as st
from streamlit_option_menu import option_menu
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
try:
    # Safely clear any active endpoints using your domain first
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except:
    pass

ngrok.kill()
os.system("pkill -f streamlit")  # Fixed shell command syntax

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give Streamlit 3 seconds to fully boot up
time.sleep(3)

# 4. Open ngrok tunnel using your specific static domain with error handling
try:
    public_url = ngrok.connect(8501, domain="relish-runt-cussed.ngrok-free.dev").public_url
except Exception as e:
    print("⚠️ Your static domain is still locked by Ngrok servers. Booting with a temporary URL instead...")
    # Fallback to standard dynamic tunnel if the static domain is stuck online
    public_url = ngrok.connect(8501).public_url

print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    os.system("pkill -f streamlit")  # Fixed shell command syntax
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")

🚀 Infosys Portal Live URL: https://relish-runt-cussed.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.


In [5]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, re, random, smtplib, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.mime.text import MIMEText

# Securely reading environment variables with your hardcoded backups
JWT_SECRET = os.environ.get('JWT_SECRET', "MyinfosysJWTSecret@2026#SsecureK")
EMAIL_ADDRESS = os.environ.get('EMAIL_ADDRESS', "samathasrikamireddy123@gmail.com")
EMAIL_PASSWORD = os.environ.get('EMAIL_PASSWORD', "rlko szxq ccom yuuf")

os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

def validate_email(email):
    pattern = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"
    return re.match(pattern, email) is not None

def validate_password(pwd):
    if len(pwd) < 8: return False
    if not re.search(r"[A-Z]", pwd): return False
    if not re.search(r"[a-z]", pwd): return False
    if not re.search(r"[0-9]", pwd): return False
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>_+-]", pwd): return False
    return True

st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-weight: 700 !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("generated_otp", None), ("otp_verified", False)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                if not email or not pwd:
                    st.error("❌ Both Email and Password fields are mandatory.")
                else:
                    with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                    if r and check_txt(pwd, r[0]):
                        st.session_state.token = make_jwt(email)
                        navigate("Dashboard")
                    else:
                        st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ All fields are mandatory.")
                elif not validate_email(email):
                    st.error("❌ Invalid Email format (e.g., must be structurally like 'ab@cd.ef').")
                elif not validate_password(pwd):
                    st.error("❌ Password must be min 8 characters, containing an uppercase, lowercase, digit, and special symbol.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or Email already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)

                if col_sq.button("Via Security Question", use_container_width=True):
                    if not email: st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            st.session_state.reset_email = email
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    if not email: st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c: r = c.execute("SELECT email FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            otp = str(random.randint(100000, 999999))
                            try:
                                msg = MIMEText(f"Your Infosys Portal password recovery OTP is: {otp}")
                                msg['Subject'] = 'Your Password Recovery Code'
                                msg['From'] = EMAIL_ADDRESS
                                msg['To'] = email

                                with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                                    server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
                                    server.sendmail(EMAIL_ADDRESS, [email], msg.as_string())

                                st.session_state.generated_otp = otp
                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.success("📩 An OTP verification code has been dispatched to your email address.")
                                time.sleep(1.5)
                                st.rerun()
                            except Exception as e:
                                st.error(f"❌ Failed to dispatch email. Error: {e}")
                        else: st.error("❌ Registered email address not found.")
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Reset Password →", use_container_width=True):
                        if not ans or not npw or not confirm_npw: st.error("⚠️ All input fields are required.")
                        elif not validate_password(npw): st.error("❌ New password does not meet structure safety criteria.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1)
                                st.session_state.reset_email = st.session_state.reset_mode = None
                                navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                elif st.session_state.get("reset_mode") == "otp":
                    if not st.session_state.otp_verified:
                        user_otp = st.text_input("Enter the 6-digit OTP code received", placeholder="123456").strip()
                        if st.button("Verify OTP Code", use_container_width=True):
                            if user_otp == st.session_state.generated_otp:
                                st.session_state.otp_verified = True
                                st.success("✅ Verification confirmed. Proceed to configure new password.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error("❌ Invalid OTP token string matched.")
                    else:
                        npw = st.text_input("New password", type="password")
                        confirm_npw = st.text_input("Confirm new password", type="password")
                        if st.button("Reset Password →", use_container_width=True):
                            if not npw or not confirm_npw: st.error("⚠️ Fields can't remain empty.")
                            elif not validate_password(npw): st.error("❌ Password fails system structure policy regulations.")
                            elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                            else:
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1)
                                st.session_state.reset_email = st.session_state.reset_mode = st.session_state.generated_otp = None
                                st.session_state.otp_verified = False
                                navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = st.session_state.reset_mode = st.session_state.generated_otp = None
                st.session_state.otp_verified = False
                navigate("Login")

else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("👥 Registered User Catalog")

        with get_db() as c:
            user_records = c.execute("SELECT username, email FROM users").fetchall()

        if user_records:
            st.markdown("""
            <table style="width:100%; border-collapse: collapse; margin-top:10px; border: 2px solid #272343;">
                <thead>
                    <tr style="background-color: #bae8e8; color: #272343; text-align: left; border-bottom: 2px solid #272343;">
                        <th style="padding: 12px; font-weight:700;">Username / Full Name</th>
                        <th style="padding: 12px; font-weight:700;">Email Address</th>
                    </tr>
                </thead>
                <tbody>
            """, unsafe_allow_html=True)

            for row in user_records:
                st.markdown(f"""
                    <tr style="border-bottom: 1px solid #bae8e8; background-color: #ffffff;">
                        <td style="padding: 12px; color: #2d334a;">{row[0]}</td>
                        <td style="padding: 12px; color: #2d334a;">{row[1]}</td>
                    </tr>
                """, unsafe_allow_html=True)

            st.markdown("</tbody></table><br>", unsafe_allow_html=True)
        else:
            st.info("No registered users present inside tracking storage environment tables.")

    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        st.plotly_chart(fig, use_container_width=True)

Writing app.py
